In [1]:
# Install dependencies for Unsloth + GPT-OSS
!pip install --upgrade -qqq uv
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" numpy pillow torchvision bitsandbytes "transformers==4.56.2" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Install openai_harmony (Harmony protocol tools)
!pip install -q openai-harmony jupyter_client pandas datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.5 MB/s eta 0:00:0000:0100:01m
Using Python 3.12.12 environment at: /usr
Resolved 5 packages in 23ms                                          
Prepared 1 package in 51ms                                               
Uninstalled 1 package in 2ms
Installed 1 package in 8ms                                  
 - trl==0.24.0
 + trl==0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 74.0 MB/s eta 0:00:00:00:01


In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

In [3]:
import kagglehub 
kagglehub.login()

In [4]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "filtered_low_pass_1_or_2.jsonl"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "barnobarno/nemotron-low-reasoning-pass-rate-1-2",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)



/tmp/ipython-input-1212032248.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 1.41G/1.41G [01:30<00:00, 16.8MB/s]


In [5]:
df.head()

,problem,expected_answer,original_expected_answer,changed_answer_to_majority,data_source,messages,metadata,license,used_in,uuid,url,user_url,user_name,tools
0,"Let \(a, b, c, d\) be four positive integers. ...",25,None,True,aops,"[{'role': 'user', 'content': 'Solve the follow...","{'reason_low_with_tool': {'count': 8, 'pass': ...",cc-by-4.0,[nano_v3],307c2636-ee59-0de3-3d3d-42364eb91df4,,,,NaN
1,Find all real solutions to the system:\n\[ x^3...,"(x,y,z)=\bigl(2\cos\theta,\;2\cos3\theta,\;2\c...",None,True,aops,"[{'role': 'user', 'content': 'Solve the follow...","{'reason_low_with_tool': {'count': 8, 'pass': ...",cc-by-4.0,[nano_v3],743b7297-896c-44e4-1a29-a5b0de45d250,,,,NaN
2,What is the total number of squares that can b...,\;\n\begin{cases}\n\displaystyle\frac{m(m+1)(3...,None,True,aops,"[{'role': 'user', 'content': 'Solve the follow...","{'reason_low_with_tool': {'count': 8, 'pass': ...",cc-by-4.0,[nano_v3],bf8daf9b-faa3-6b0c-a8ec-85a504a12694,,,,NaN
3,Given the function \( f(x) = \frac{2x}{1 + x^2...,\;0\le\displaystyle\sum_{k=1}^{n}f^{-1}(x_{k})...,None,True,aops,"[{'role': 'user', 'content': 'Solve the follow...","{'reason_low_with_tool': {'count': 8, 'pass': ...",cc-by-4.0,[nano_v3],5a48c774-c2e5-95ed-c516-068924c7d85e,,,,NaN
4,"In triangle \(ABC\), the altitude, angle bisec...",22.5^{\circ},None,True,aops,"[{'role': 'user', 'content': 'Solve the follow...","{'reason_low_with_tool': {'count': 8, 'pass': ...",cc-by-4.0,[nano_v3],a79c89d1-67a5-aa60-957d-edc59fb8acf5,,,,NaN


In [78]:
train_data = df.copy()
train_data.drop(columns=["uuid","original_expected_answer","license" ,"used_in" ,"user_name" ,"user_url" ,"url"], inplace=True ,axis=1)
#train_data.dropna(inplace=True)
train_data = train_data[train_data["tools"].isna()]
train_data.drop(columns=["tools"], inplace=True ,axis=1)

In [79]:
train_data.head()

,problem,expected_answer,changed_answer_to_majority,data_source,messages,metadata
0,"Let \(a, b, c, d\) be four positive integers. ...",25,True,aops,"[{'role': 'user', 'content': 'Solve the follow...","{'reason_low_with_tool': {'count': 8, 'pass': ..."
1,Find all real solutions to the system:\n\[ x^3...,"(x,y,z)=\bigl(2\cos\theta,\;2\cos3\theta,\;2\c...",True,aops,"[{'role': 'user', 'content': 'Solve the follow...","{'reason_low_with_tool': {'count': 8, 'pass': ..."
2,What is the total number of squares that can b...,\;\n\begin{cases}\n\displaystyle\frac{m(m+1)(3...,True,aops,"[{'role': 'user', 'content': 'Solve the follow...","{'reason_low_with_tool': {'count': 8, 'pass': ..."
3,Given the function \( f(x) = \frac{2x}{1 + x^2...,\;0\le\displaystyle\sum_{k=1}^{n}f^{-1}(x_{k})...,True,aops,"[{'role': 'user', 'content': 'Solve the follow...","{'reason_low_with_tool': {'count': 8, 'pass': ..."
4,"In triangle \(ABC\), the altitude, angle bisec...",22.5^{\circ},True,aops,"[{'role': 'user', 'content': 'Solve the follow...","{'reason_low_with_tool': {'count': 8, 'pass': ..."


In [81]:
train_data.shape

(154123, 6)

In [82]:
train_data["messages"].iloc[0]

[{'role': 'user',
  'content': 'Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}.\n\nLet \\(a, b, c, d\\) be four positive integers. The integer \\(n\\) is the smallest positive integer such that the sum of every rational number of the form \\(\\frac{1}{a^2 + b^3 + c^8 + d^n}\\) does not exceed an integer \\(M\\). What is \\(n\\)?',
  'reasoning_content': None},
 {'role': 'assistant',
  'content': 'To make the sum  \n\n\\[\n\\sum_{a,b,c,d\\ge 1}\\frac{1}{a^{2}+b^{3}+c^{8}+d^{\\,n}}\n\\]\n\nfinite we need the corresponding multiple integral to converge:\n\n\\[\n\\int_{1}^{\\infty}\\!\\!\\int_{1}^{\\infty}\\!\\!\\int_{1}^{\\infty}\\!\\!\\int_{1}^{\\infty}\n\\frac{dx\\,dy\\,dz\\,dw}{x^{2}+y^{3}+z^{8}+w^{\\,n}} .\n\\]\n\nA known criterion for convergence of such integrals is  \n\n\\[\n\\sum_{i=1}^{k}\\frac{1}{p_i}<1,\n\\]\n\nwhere the denominator is a sum of powers \\(x_i^{p_i}\\).  \nHere \\(p_1=2,\\;p_2=3,\\;p_3=8,\\;p_4=n\\). Hence we requ

In [83]:
def remove_none_keys(messages):
    return [{k: v for k, v in entry.items() if v is not None} for entry in messages]

In [84]:
def replace_none_with_null(messages):
    return [{k: v if v is not None else "null" for k, v in entry.items()} for entry in messages]

In [85]:
def format_for_gpt_oss(example):
    messages = example['messages']
    new_messages = []
    
    for msg in messages:
        new_msg = msg.copy()
        
        # 1. Rename 'reasoning_content' to 'thinking'
        if 'reasoning_content' in new_msg:
            new_msg['thinking'] = new_msg.pop('reasoning_content')
        
        # 2. Ensure intermediate tool steps don't conflict
        # The template raises an error if you have BOTH 'thinking' and 'content' 
        # inside a tool call message. Your data has content='', which is fine, 
        # but purely safe practice is to ensure it is None or empty.
        if new_msg.get('tool_calls') and new_msg.get('thinking'):
             new_msg['content'] = "" # Ensure this is empty to avoid template error

        new_messages.append(new_msg)
    
    return {'messages': new_messages}

# Apply to your dataset
train_data_formatted = train_data.apply(format_for_gpt_oss, axis=1)
train_data["messages"] = train_data_formatted


In [86]:
train_data["messages"].iloc[0]["messages"]
train_data["AA"] = train_data["messages"].apply(lambda x: x["messages"])

In [87]:
train_data["AA"].iloc[0]

[{'role': 'user',
  'content': 'Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}.\n\nLet \\(a, b, c, d\\) be four positive integers. The integer \\(n\\) is the smallest positive integer such that the sum of every rational number of the form \\(\\frac{1}{a^2 + b^3 + c^8 + d^n}\\) does not exceed an integer \\(M\\). What is \\(n\\)?',
  'thinking': None},
 {'role': 'assistant',
  'content': 'To make the sum  \n\n\\[\n\\sum_{a,b,c,d\\ge 1}\\frac{1}{a^{2}+b^{3}+c^{8}+d^{\\,n}}\n\\]\n\nfinite we need the corresponding multiple integral to converge:\n\n\\[\n\\int_{1}^{\\infty}\\!\\!\\int_{1}^{\\infty}\\!\\!\\int_{1}^{\\infty}\\!\\!\\int_{1}^{\\infty}\n\\frac{dx\\,dy\\,dz\\,dw}{x^{2}+y^{3}+z^{8}+w^{\\,n}} .\n\\]\n\nA known criterion for convergence of such integrals is  \n\n\\[\n\\sum_{i=1}^{k}\\frac{1}{p_i}<1,\n\\]\n\nwhere the denominator is a sum of powers \\(x_i^{p_i}\\).  \nHere \\(p_1=2,\\;p_2=3,\\;p_3=8,\\;p_4=n\\). Hence we require  \n\n

In [88]:
dataset = train_data.iloc[0:10].copy()
dataset["AA"] = dataset["AA"].apply(remove_none_keys)

dataset["text"] = dataset.apply(lambda row: tokenizer.apply_chat_template(
    row["AA"], 
    tokenize=False, 
    add_generation_prompt=True,
    reasoning_effort="low"
), axis=1)


In [89]:
dataset["text"].iloc[0]

"<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2026-01-16\n\nReasoning: low\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.\nCalls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}.\n\nLet \\(a, b, c, d\\) be four positive integers. The integer \\(n\\) is the smallest positive integer such that the sum of every rational number of the form \\(\\frac{1}{a^2 + b^3 + c^8 + d^n}\\) does not exceed an integer \\(M\\). What is \\(n\\)?<|end|><|start|>assistant<|channel|>final<|message|>To make the sum  \n\n\\[\n\\sum_{a,b,c,d\\ge 1}\\frac{1}{a^{2}+b^{3}+c^{8}+d^{\\,n}}\n\\]\n\nfinite we need the corresponding multiple integral to converge:\n\n\\[\n\\int_{1}^{\\infty}\\!\\!\\int_{1}^{\\infty}\\!\\!\\int_{1}^{\\infty}\\!\\!\\int_{1}

In [90]:
from datasets import Dataset

# Convert pandas DataFrame to HuggingFace Dataset
hf_dataset = Dataset.from_pandas(dataset)

In [91]:
# Add LoRA adapters with rank 16
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

RuntimeError: Unsloth: You already added LoRA adapters to your model!

In [92]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        #num_train_epochs=0.25, 
        max_steps=10 ,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

num_proc must be <= 10. Reducing num_proc to 10 for dataset of size 10.


Unsloth: Tokenizing ["text"] (num_proc=10):   0%|          | 0/10 [00:00<?, ? examples/s]

In [93]:
from unsloth.chat_templates import train_on_responses_only

gpt_oss_kwargs = dict(
    instruction_part="<|start|>user<|message|>", 
    response_part="<|start|>assistant<|channel|>final<|message|>"
)

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)

num_proc must be <= 10. Reducing num_proc to 10 for dataset of size 10.


Map (num_proc=10):   0%|          | 0/10 [00:00<?, ? examples/s]

In [94]:
trainer_stats = trainer.train()

# Keep memory stats if needed
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Peak reserved memory = {used_memory} GB")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 5 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 7,962,624 of 20,922,719,808 (0.04% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.446800
2,0.411700
3,0.464100
4,0.316600
5,0.416600
6,0.298600
7,0.363600
8,0.303200
9,0.286700
10,0.416100


Peak reserved memory = 16.059 GB


In [75]:
text = tokenizer.apply_chat_template(
    dataset["AA"].iloc[0],
    tokenize = False,
    add_generation_prompt = True,
    reasoning_effort = "low",
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1.0,
    max_new_tokens = 512,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<|channel|>analysis<|message|>We derived that V = 152√3/5 π ~ 165.4185. Ensure cylindrical shells method gives same. Shells about x-axis: integrate in y from intersection y-values. For a given y, the horizontal strip extends between leftmost and rightmost x-values of the region: which are from parabola (right) and line (left). For y between y1=5-2√3 and y2=5+2√3.

Find x from parabola: x = ±√(y-1). Intersection region includes both sides? The region bounded by the curves is between the two curves, but need to identify which is left/right. For a fixed y, line gives x = (3 - y)/2 (since y = -2x + 3). Parabola gives x = ±√(y-1). For y between 1 and 5+2√3 the curve intersects: line above parabola.

But for rotation about x-axis with shells, radius = y, length = x_right - x_left. Right boundary is from line? Actually line is to the left of parabola for given y in that range? Let's test: y=5: line gives x = (3-5)/2 = -1. Parabola x=±√(4)=±2. The region bounded by the two curves and by the li

In [76]:
text

'<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2026-01-16\n\nReasoning: low\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.\nCalls to these tools must go to the commentary channel: \'functions\'.<|end|><|start|>user<|message|>Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}.\n\nWhat is the volume of the solid made by rotating the region bounded by $y=x^2+1$ and $y=-2x+3$ about the $x$-axis using the cylindrical shells method?<|end|><|start|>assistant to=functions.stateful_python_code_exec<|channel|>commentary json<|message|>"{\\"code\\":\\"import sympy as sp\\\\nx=sp.symbols(\'x\')\\\\nsol=sp.solve(x**2+2*x-2, x)\\\\nsol\\"}"<|call|><|start|>functions.stateful_python_code_exec to=assistant<|channel|>commentary<|message|>"[-1 + sqrt(3), -sqrt(3) - 1]"<|end|><|start|>assistant to=functions.stateful_python_code_exec<

In [ ]:
model.save_pretrained("gpt_oss_20b_nemotron_finetuned")
tokenizer.save_pretrained("gpt_oss_20b_nemotron_finetuned")
print("Model saved to 'gpt_oss_20b_nemotron_finetuned'")